# 01 — Data Exploration

Load the two datasets and characterise them: row counts, feature counts,
label balance, attack-window visualisations, and per-feature statistics.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_hai, load_morris, morris_train_test_split
from src.utils import set_seed, save_figure

set_seed(42)
sns.set_theme(style='whitegrid', context='notebook')


## HAI 21.03

In [ ]:
hai_train, hai_test = load_hai()
print(f'HAI train: {hai_train.shape}, test: {hai_test.shape}')
print(f'HAI test attack rate: {hai_test["label"].mean():.4f}')
print(f'HAI feature count: {len([c for c in hai_train.columns if c != "label"])}')
hai_train.head(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
pd.Series({'Train (normal)': len(hai_train), 'Test': len(hai_test)}).plot(
    kind='bar', ax=axes[0], color=['#2b8cbe', '#e34a33']
)
axes[0].set_title('HAI row counts')
axes[0].set_ylabel('rows')

hai_test['label'].value_counts().rename({0: 'Normal', 1: 'Attack'}).plot(
    kind='bar', ax=axes[1], color=['#31a354', '#de2d26']
)
axes[1].set_title('HAI test label balance')
axes[1].set_ylabel('rows')
plt.tight_layout()
save_figure(fig, 'hai_overview', subdir='01_exploration')
plt.show()


In [ ]:
sensor_cols = [c for c in hai_train.columns if c != 'label'][:6]
fig, axes = plt.subplots(3, 2, figsize=(11, 7), sharex=False)
for ax, col in zip(axes.flat, sensor_cols):
    ax.plot(hai_train[col].to_numpy()[:8000], color='#2b8cbe', linewidth=0.6)
    ax.set_title(col)
    ax.set_xlabel('sample')
plt.suptitle('HAI sensor traces (first 8000 normal samples)', y=1.02)
plt.tight_layout()
save_figure(fig, 'hai_sensor_traces', subdir='01_exploration')
plt.show()


In [ ]:
# Attack-window visualisation: plot one sensor over the test set, colouring attack windows.
t_col = hai_test.columns[0]
series = hai_test[t_col].to_numpy()[:60000]
labels = hai_test['label'].to_numpy()[:60000]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(series, color='#2b8cbe', linewidth=0.5, label='sensor')
ax.fill_between(
    np.arange(len(series)), series.min(), series.max(),
    where=labels.astype(bool), color='#de2d26', alpha=0.25, label='attack'
)
ax.legend(loc='upper right')
ax.set_title(f'HAI test — {t_col} with attack windows highlighted')
plt.tight_layout()
save_figure(fig, 'hai_attack_windows', subdir='01_exploration')
plt.show()


## Morris Gas Pipeline

In [ ]:
morris = load_morris()
print(f'Morris shape: {morris.shape}')
print(f'Morris attack rate: {morris["label"].mean():.4f}')
print(f'Morris feature count: {len([c for c in morris.columns if c != "label"])}')
morris.head(3)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
morris['label'].value_counts().rename({0: 'Normal', 1: 'Attack'}).plot(
    kind='bar', ax=ax, color=['#31a354', '#de2d26']
)
ax.set_title('Morris label balance')
ax.set_ylabel('rows')
plt.tight_layout()
save_figure(fig, 'morris_label_balance', subdir='01_exploration')
plt.show()


In [ ]:
# Correlation heatmap of Morris features (post-leak-removal).
feats = [c for c in morris.columns if c != 'label']
corr = morris[feats].corr().fillna(0)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, cmap='RdBu_r', center=0, ax=ax, square=True,
            xticklabels=True, yticklabels=True, cbar_kws={'shrink': 0.6})
ax.set_title('Morris feature correlation')
plt.tight_layout()
save_figure(fig, 'morris_correlation', subdir='01_exploration')
plt.show()


In [ ]:
# Descriptive statistics summary (thesis table material).
summary = pd.DataFrame({
    'dataset': ['HAI 21.03', 'Morris gas pipeline'],
    'rows_total': [len(hai_train) + len(hai_test), len(morris)],
    'features': [len([c for c in hai_train.columns if c != 'label']),
                 len([c for c in morris.columns if c != 'label'])],
    'attack_rate': [hai_test['label'].mean(), morris['label'].mean()],
    'modality': ['continuous sensor time-series', 'Modbus RTU tabular packets'],
})
summary
